In [1]:
# IMPORT LIBRARIES

In [2]:
import pandas as pd
import sqlite3

In [3]:
# CONNECT DATABASE

In [6]:
conn = sqlite3.connect('ipl_data.db')
print("Database connected successfully")

Database connected successfully


In [7]:
# ROW_NUMBER — RANK RCB BATSMEN BY TOTAL RUNS

In [8]:
Query = """
SELECT  batter,
        SUM(batsman_runs) as total_runs,
        ROW_NUMBER() OVER (ORDER BY SUM(batsman_runs) DESC) as rank
FROM deliveries
where batting_team LIKE 'Royal Challengers%'
GROUP BY batter
LIMIT 10
"""
ranked_batsman = pd.read_sql_query(Query,conn)
print("Top 10 RCB batsmen by runs (ROW_NUMBER):")
print(ranked_batsmen)

Top 10 RCB batsmen by runs (ROW_NUMBER):


NameError: name 'ranked_batsmen' is not defined

In [ ]:
# RANK vs DENSE_RANK COMPARISON

In [9]:
query = """
SELECT 
    batter,
    total_runs,
    RANK() OVER (ORDER BY total_runs DESC) AS rank_normal,
    DENSE_RANK() OVER (ORDER BY total_runs DESC) AS rank_dense
FROM (
    SELECT 
        batter,
        SUM(batsman_runs) AS total_runs
    FROM deliveries d
    JOIN matches m ON d.match_id = m.id
    WHERE (m.team1 LIKE 'Royal Challengers%'
        OR m.team2 LIKE 'Royal Challengers %')
      AND d.batting_team LIKE 'Royal Challengers%'
    GROUP BY batter
)
LIMIT 20
"""

rank_comparison = pd.read_sql_query(query, conn)
print("RANK vs DENSE_RANK comparison:")
print(rank_comparison)

RANK vs DENSE_RANK comparison:
            batter  total_runs  rank_normal  rank_dense
0          V Kohli        8014            1           1
1   AB de Villiers        4510            2           2
2         CH Gayle        3175            3           3
3     F du Plessis        1636            4           4
4       GJ Maxwell        1266            5           5
5        JH Kallis        1132            6           6
6       KD Karthik         937            7           7
7         R Dravid         898            8           8
8       D Padikkal         884            9           9
9       RM Patidar         799           10          10
10        PA Patel         731           11          11
11   Mandeep Singh         597           12          12
12      TM Dilshan         587           13          13
13      RV Uthappa         549           14          14
14     LRPL Taylor         517           15          15
15       SS Tiwary         487           16          16
16      MA Agarwa

In [10]:
# TOP 3 RCB BATSMEN PER SEASON (PARTITION BY)

In [11]:
Query = """
SELECT  *
FROM (select
         batter,
        season,
        SUM(batsman_runs) AS total_runs,
        RANK() OVER (PARTITION BY season ORDER BY SUM(batsman_runs) DESC) AS rank
    FROM deliveries
    JOIN matches ON matches.id = deliveries.match_id
    where batting_team LIKE 'Royal Challengers%'
    GROUP BY batter,season
) as ranked_data
where rank <=3
order by season,rank
"""
top3_per_season = pd.read_sql_query(Query, conn)
print("Top 3 RCB batsmen per season:")
print(top3_per_season.head(20))

Top 3 RCB batsmen per season:
            batter   season  total_runs  rank
0         R Dravid  2007/08         371     1
1       MV Boucher  2007/08         225     2
2        JH Kallis  2007/08         199     3
3        JH Kallis     2009         361     1
4      LRPL Taylor     2009         280     2
5         R Dravid     2009         271     3
6        JH Kallis  2009/10         572     1
7       RV Uthappa  2009/10         374     2
8          V Kohli  2009/10         307     3
9         CH Gayle     2011         608     1
10         V Kohli     2011         557     2
11  AB de Villiers     2011         312     3
12        CH Gayle     2012         733     1
13         V Kohli     2012         364     2
14  AB de Villiers     2012         319     3
15        CH Gayle     2013         720     1
16         V Kohli     2013         639     2
17  AB de Villiers     2013         373     3
18  AB de Villiers     2014         395     1
19    Yuvraj Singh     2014         376     2


In [12]:
# KOHLI'S CUMULATIVE RUNS ACROSS SEASONS

In [13]:
Query = """
SELECT  *
FROM (select
        season,
        SUM(batsman_runs) AS total_runs,
        SUM(SUM(batsman_runs)) OVER (ORDER BY season) as cumulative_runs
    FROM deliveries
    JOIN matches ON matches.id = deliveries.match_id
    where batter LIKE 'V Kohli'
    GROUP BY season
) as grouped_data
order by season
"""
kohli_cumulative = pd.read_sql_query(Query, conn)
print("Kohli's cumulative RCB runs season by season:")
print(kohli_cumulative)

Kohli's cumulative RCB runs season by season:
     season  total_runs  cumulative_runs
0   2007/08         165              165
1      2009         246              411
2   2009/10         307              718
3      2011         557             1275
4      2012         364             1639
5      2013         639             2278
6      2014         359             2637
7      2015         505             3142
8      2016         973             4115
9      2017         308             4423
10     2018         530             4953
11     2019         464             5417
12  2020/21         471             5888
13     2021         405             6293
14     2022         341             6634
15     2023         639             7273
16     2024         741             8014


In [14]:
# TOP 3 RCB BOWLERS PER SEASON (PARTITION BY)

In [15]:
Query = """
SELECT  *
FROM (select
         bowler,
        season,
        SUM(is_wicket) AS total_runs,
        RANK() OVER (PARTITION BY season ORDER BY SUM(is_wicket) DESC) AS rank
    FROM deliveries
    JOIN matches ON matches.id = deliveries.match_id
    where bowling_team LIKE 'Royal Challengers%'
    GROUP BY bowler,season
) as ranked_data
where rank <=3
"""
top3_bowlers = pd.read_sql_query(Query, conn)
print("Top 3 RCB bowlers per season:")
print(top3_bowlers.head(20))

Top 3 RCB bowlers per season:
              bowler   season  total_runs  rank
0             Z Khan  2007/08          15     1
1           DW Steyn  2007/08          13     2
2            P Kumar  2007/08          12     3
3           A Kumble     2009          22     1
4            P Kumar     2009          16     2
5      R Vinay Kumar     2009           9     3
6   RE van der Merwe     2009           9     3
7           A Kumble  2009/10          19     1
8      R Vinay Kumar  2009/10          19     1
9           DW Steyn  2009/10          17     3
10         S Aravind     2011          22     1
11            Z Khan     2011          15     2
12        DL Vettori     2011          13     3
13     R Vinay Kumar     2012          21     1
14            Z Khan     2012          21     1
15    M Muralitharan     2012          15     3
16     R Vinay Kumar     2013          27     1
17         R Rampaul     2013          15     2
18          RP Singh     2013          14     3
19        

In [16]:
# CLOSE CONNECTION

In [17]:
conn.close()
print("Connection closed.")

Connection closed.


## Day 10 Summary: Window Functions

 Completed
- Ranked RCB batsmen using ROW_NUMBER(), RANK(), DENSE_RANK()
- Compared RANK vs DENSE_RANK behavior with ties
- Used PARTITION BY to rank players within each season
- Calculated running (cumulative) totals for Kohli's career
- Ranked bowlers by wickets within each season

 Key Queries Learned
- ROW_NUMBER() OVER (ORDER BY ...)
- RANK() and DENSE_RANK() — and how they handle ties
- PARTITION BY for grouping within window functions
- SUM() OVER (ORDER BY ...) for running totals
- Subquery + window function pattern (aggregate first, then window)

 Next
Day 11 — CTEs: Common Table Expressions for cleaner, readable queries